# 水力発電候補地選定システム

## このシステムでできること

地名を入力するだけで、その地域で**水力発電に適した場所**を自動で探し出します。

- **地図上に候補地を表示**（水源・取水口・発電所の位置）
- **発電量を自動計算**（どのくらいの電力が作れるか）
- **グラフで比較**（複数の候補地を見やすく比較）
- **結果をファイル保存**（CSV・HTML・PNG形式）

## 使い方（3ステップ）

### ステップ1：地名を入力
```python
selector = HydroSiteSelector("松本市")
```
※ 日本国内の市区町村名を入力してください

### ステップ2：分析を実行
```python
map_result, fig_result = selector.run_analysis()
```
※ 実行時間の目安：数分～十数分程度（地域の広さによって変わります）

### ステップ3：結果を確認
- **地図**: 上位3組の候補地が色分けされて表示されます
  - 赤色：第1位の組合せ
  - 青色：第2位の組合せ
  - 緑色：第3位の組合せ
- **グラフ**: 標高分布や発電量の比較が表示されます
- **データ**: CSV形式で詳細データが保存されます

## システムの仕組み

### 探索範囲の決定方法

1. **正確な行政区画境界を取得**
   - OpenStreetMapから行政区画の境界ポリゴンを取得
   - 数万点の頂点データで正確な範囲を定義
   - 面積誤差は通常1%未満の高精度

2. **境界内のみを分析**
   - グリッドポイントを境界内にフィルタリング
   - 河川データも境界と交差するもののみを使用
   - 隣接自治体のデータは除外

### 候補地の選定方法（詳細）

システムは3段階で候補地を絞り込んでいきます：

#### 第1段階：グリッドポイントの生成と評価

1. **グリッドの生成**
   - 探索範囲を格子状に分割（例：28×28 = 784点）
   - 行政区画の境界内のポイントのみを残す（フィルタリング）
   - 実際の候補点数は境界形状によって変動（通常30～60%が境界内）

2. **標高データの取得**
   - 各グリッドポイントの標高をOpen-Elevation APIで取得
   - バッチ処理で高速化（150点ずつまとめて取得）
   - 標高範囲の把握と地形の理解

3. **地形勾配の計算**
   - 各ポイントの周囲8方向の標高差を計算
   - 勾配の大きさ：地形の急峻さを示す
   - 勾配の向き：水の流れる方向を示す

#### 第2段階：各施設タイプの候補地選定

**1. 水源候補地の選定**

評価基準（スコアが高いほど良い）：

```
水源スコア = (標高の高さ) × 1.0 + (推定流量) × 0.3 + (勾配の大きさ) × 0.2
```

- **標高の重視**: 高い場所ほど大きな落差が得られる
- **流量の考慮**: 近くの河川から推定した流量を加点
- **勾配の活用**: 適度な勾配は水が集まりやすいことを示す
- **選定数**: 指定した数（デフォルト40箇所）の上位候補を選定

**河川流量の推定方法**:
1. 各ポイントから最も近い河川を探索（通常5km以内）
2. 河川タイプによる基準流量
   - `river`（大河川）: 1.0 m³/s
   - `stream`（小河川）: 0.5 m³/s
3. 距離による減衰計算
   - 減衰率 = exp(-距離 / 2.0)
   - 遠い河川ほど影響が小さくなる

**2. 取水口候補地の選定**

評価基準：

```
取水口スコア = (標高の中程度さ) × 1.0 + (勾配の緩やかさ) × 0.5
```

- **標高の中間性**: 水源より低く、発電所より高い中間標高を優先
- **勾配の緩やかさ**: 建設しやすい平坦な場所を優先
  - 逆数スコアを使用（勾配が小さいほど高得点）
- **選定数**: 水源と同数（デフォルト40箇所）

**3. 発電所候補地の選定**

評価基準：

```
発電所スコア = (標高の低さ) × 1.0 + (勾配の緩やかさ) × 0.5
```

- **標高の低さ**: 低い場所ほど大きな落差が得られる
  - 最大標高 - 現在標高 でスコア化
- **勾配の緩やかさ**: 建設・保守が容易な平坦地を優先
- **選定数**: 水源と同数（デフォルト40箇所）

#### 第3段階：組み合わせの評価と厳選

**全組み合わせの評価**:
- 水源40 × 取水口40 × 発電所40 = 64,000通りの組み合わせを評価
- すべての組み合わせについて発電量を計算（全探索）

**1. 物理的制約のチェック**

各組み合わせが以下の条件を満たすか確認：

```python
# 標高の順序
水源の標高 > 取水口の標高 > 発電所の標高

# 有効落差の確認
有効落差 = 水源標高 - 発電所標高 > 0

# 距離の制約（オプション）
施設間距離が現実的な範囲内
```

**2. 発電量の計算**

物理的制約を満たす組み合わせについて発電量を計算：

```python
# 理論発電量の計算
理論発電量 = 9.8 × 流量(m³/s) × 有効落差(m)

# 効率損失の考慮
- 管路損失: 約10%（摩擦・曲がりなどによる）
- 水車効率: 約85%（水力→回転力の変換）
- 発電機効率: 約95%（回転力→電力の変換）

# 総合効率
総合効率 = (1 - 0.10) × 0.85 × 0.95 ≈ 0.73

# 実際の発電量
実発電量(kW) = 理論発電量 × 0.73
```

簡易計算では総合効率を0.8として概算：

$$発電量(kW) = 9.8 \times 流量(m³/s) \times 有効落差(m) \times 0.8$$

**3. ランキングと上位選出**

- すべての有効な組み合わせを発電量で降順ソート
- 上位N組（デフォルト30組）を最終候補として選出
- 発電量・有効落差・距離などの詳細データを記録

**選出される組み合わせの特徴**:
- 高標高の水源 × 低標高の発電所 = 大きな落差
- 河川に近い水源 = 豊富な流量
- 適度な施設間距離 = 建設コストの現実性

### 最適化のポイント

**なぜ全探索なのか？**:
- 候補数を事前に絞り込んでいるため、組み合わせ数は現実的な範囲
- 全探索により真の最適解を保証
- 計算時間は数秒～数十秒程度（許容範囲内）

**もし計算が遅い場合**:
```python
# 候補数を減らして高速化
selector.run_analysis(candidates_per_type=20)  # 20³ = 8,000通り
```

### 発電量の計算方法

$$発電量(kW) = 9.8 \times 流量(m³/s) \times 有効落差(m) \times 効率$$

- **流量**: 探索範囲内の実際の河川データから推定（通常 0.3～0.5 m³/s程度）
- **有効落差**: 水源の標高 - 発電所の標高（通常 1000～2000m程度）
- **効率**: 水車や発電機の総合効率（約80%）

### 河川データの取得方法

システムは探索範囲内の河川データを自動的に取得します：

1. **Overpass APIから河川情報を取得**
   - OpenStreetMapの河川データ（river, stream）を検索
   - 境界ポリゴンと交差する河川のみを抽出
   - 通常、数千～数万本の河川を検出

2. **流量の推定**
   - 河川タイプ（river: 大河川、stream: 小河川）から基準流量を設定
   - 各候補地点から最も近い河川を探索
   - 距離による減衰を考慮して流量を推定

## 算出手順の詳細（仕様）

### ステップ1：探索範囲の確定

**目的**: 分析対象地域を正確に定義する

**手順**:
1. ユーザーが入力した地名（例：「長野県」）をNominatim APIで検索
2. 該当する行政区画の中心座標（緯度・経度）を取得
3. 行政区画の境界ポリゴンデータを取得
   - 境界は数千～数万個の座標点で構成される多角形
   - 例：長野県の場合、約36,000個の頂点で境界を定義
4. 境界ポリゴンから面積を計算
   - Shapelyライブラリで多角形の面積を算出
   - 緯度1度あたりの距離（約111km）を使って実面積（km²）に変換

**出力データ**:
- 中心座標：(緯度, 経度)
- 境界ポリゴン：[(lat1, lon1), (lat2, lon2), ...]
- バウンディングボックス：(最小緯度, 最大緯度, 最小経度, 最大経度)
- 面積：○○ km²

---

### ステップ2：探索グリッドの生成

**目的**: 地域全体を均等にカバーする評価ポイントを配置

**手順**:
1. **グリッドサイズの決定**
   - 自動モード：面積 ÷ 目標密度(15 km²/点) の平方根
   - 手動指定：ユーザーが指定した値を使用
   - 例：面積が13,500 km²の場合 → √(13500/15) = 30 → 30×30グリッド

2. **グリッドポイントの配置**
   - バウンディングボックス内を等間隔に分割
   - 緯度方向にN個、経度方向にN個の点を配置
   - 総候補点数 = N × N 個

3. **境界内フィルタリング**
   - 各グリッドポイントが境界ポリゴン内に含まれるかチェック
   - Shapely の `polygon.contains(point)` で判定
   - 境界外の点は除外（通常30～60%が除外される）

**出力データ**:
- 境界内グリッドポイント：[(lat1, lon1), (lat2, lon2), ...] 通常100～500点程度

---

### ステップ3：標高データの取得

**目的**: 各グリッドポイントの標高を取得し、地形を把握

**手順**:
1. **API呼び出しの準備**
   - グリッドポイントを150点ずつのバッチに分割
   - Open-Elevation APIは1回のリクエストで複数点の標高を返す

2. **標高データの取得**
   - 各バッチについてAPIリクエストを送信
   - 座標(緯度, 経度)を送り、標高(メートル)を受信
   - リトライ機能：タイムアウト時は再試行

3. **標高データの整理**
   - グリッドポイントと標高を対応付け
   - 標高の統計量を計算（最小値、最大値、平均値、中央値）

**出力データ**:
- 標高配列：[elev1, elev2, elev3, ...] 各グリッドポイントに対応
- 標高統計：最小○○m、最大○○m、平均○○m

---

### ステップ4：河川データの取得

**目的**: 水源候補地の流量を推定するための河川情報を収集

**手順**:
1. **Overpass APIクエリの構築**
   ```
   [way["waterway"~"river|stream"](poly:"境界ポリゴンの座標列")]
   ```
   - 河川タイプを「river」（大河川）と「stream」（小河川）に限定
   - 境界ポリゴンと交差する河川のみを取得

2. **河川データの解析**
   - 各河川の座標リスト（緯度・経度の配列）を取得
   - 河川のタイプ（river/stream）を記録
   - 河川名があれば記録

3. **境界内フィルタリング**
   - 河川の座標が境界ポリゴンと交差するかチェック
   - Shapely の `polygon.intersects(linestring)` で判定
   - 交差しない河川は除外

4. **河川の統計**
   - タイプ別の河川数を集計
   - 平均流量の推定（後述）

**出力データ**:
- 河川リスト：[{type: "river", coords: [...], name: "..."}, ...]
- 河川数：river: ○○本、stream: ○○本、合計○○本

---

### ステップ5：地形勾配の計算

**目的**: 水の流れやすさ、建設の難易度を評価

**手順**:
1. **隣接点の標高差を計算**
   - 各グリッドポイントについて、周囲8方向の隣接点を探索
   - 隣接点との標高差を計算
   - 距離で除算して勾配（m/m）を算出

2. **勾配の合成**
   - 8方向の勾配ベクトルを合成
   - 合成勾配の大きさ：地形の急峻さ
   - 合成勾配の向き：水が流れる方向

3. **勾配の正規化**
   - 全ポイントの勾配を0～1の範囲に正規化
   - 最小勾配 = 0（平坦）、最大勾配 = 1（急峻）

**出力データ**:
- 勾配配列：[grad1, grad2, ...] 各グリッドポイントに対応
- 勾配範囲：最小○○、最大○○

---

### ステップ6：水源候補地の選定

**目的**: 高標高・高流量の地点を水源候補として選定

**手順**:
1. **流量の推定**（各グリッドポイントについて）
   - 最も近い河川を探索（通常5km以内）
   - 河川までの距離 d (km) を計算
   - 河川タイプから基準流量を設定
     - river: Q_base = 1.0 m³/s
     - stream: Q_base = 0.5 m³/s
   - 距離減衰を適用：Q = Q_base × exp(-d / 2.0)
   - 河川が見つからない場合：Q = 0.1 m³/s（最小値）

2. **水源スコアの計算**（各グリッドポイントについて）
   ```
   正規化標高 = (標高 - 最小標高) / (最大標高 - 最小標高)
   正規化流量 = (流量 - 最小流量) / (最大流量 - 最小流量)
   正規化勾配 = (勾配 - 最小勾配) / (最大勾配 - 最小勾配)
   
   水源スコア = 正規化標高 × 1.0 + 正規化流量 × 0.3 + 正規化勾配 × 0.2
   ```
   - 重み付けの意味：
     - 標高（×1.0）：最重要要素。高いほど落差が大きい
     - 流量（×0.3）：水量の確保。多いほど発電量が増える
     - 勾配（×0.2）：水の集まりやすさ。適度な勾配が望ましい

3. **上位候補の選定**
   - すべてのグリッドポイントをスコアで降順ソート
   - 上位N個（デフォルト40個）を水源候補として選定

**出力データ**:
- 水源候補リスト：[{lat, lon, elevation, flow, score}, ...]
- 候補数：○○箇所
- 平均標高：○○m、平均流量：○○ m³/s

---

### ステップ7：取水口候補地の選定

**目的**: 中間標高・平坦地を取水口候補として選定

**手順**:
1. **取水口スコアの計算**（各グリッドポイントについて）
   ```
   中間度 = 1 - |2 × 正規化標高 - 1|
     # 標高が中間(0.5)に近いほど1に近づく
     # 例：標高0.5 → 中間度1.0、標高0.0 or 1.0 → 中間度0.0
   
   平坦度 = 1 / (1 + 正規化勾配)
     # 勾配が小さいほど1に近づく
     # 例：勾配0 → 平坦度1.0、勾配∞ → 平坦度0.0
   
   取水口スコア = 中間度 × 1.0 + 平坦度 × 0.5
   ```
   - 重み付けの意味：
     - 中間度（×1.0）：水源と発電所の中間標高が理想
     - 平坦度（×0.5）：建設しやすさ、保守の容易さ

2. **上位候補の選定**
   - すべてのグリッドポイントをスコアで降順ソート
   - 上位N個（水源と同数）を取水口候補として選定

**出力データ**:
- 取水口候補リスト：[{lat, lon, elevation, score}, ...]
- 候補数：○○箇所
- 平均標高：○○m

---

### ステップ8：発電所候補地の選定

**目的**: 低標高・平坦地を発電所候補として選定

**手順**:
1. **発電所スコアの計算**（各グリッドポイントについて）
   ```
   低標高度 = 1 - 正規化標高
     # 標高が低いほど1に近づく
     # 例：最低標高 → 1.0、最高標高 → 0.0
   
   平坦度 = 1 / (1 + 正規化勾配)
     # 取水口と同じ計算
   
   発電所スコア = 低標高度 × 1.0 + 平坦度 × 0.5
   ```
   - 重み付けの意味：
     - 低標高度（×1.0）：低いほど落差が大きい
     - 平坦度（×0.5）：建設・保守の容易さ

2. **上位候補の選定**
   - すべてのグリッドポイントをスコアで降順ソート
   - 上位N個（水源と同数）を発電所候補として選定

**出力データ**:
- 発電所候補リスト：[{lat, lon, elevation, score}, ...]
- 候補数：○○箇所
- 平均標高：○○m

---

### ステップ9：組み合わせの評価

**目的**: すべての組み合わせについて発電量を計算し、ランク付け

**手順**:
1. **組み合わせの生成**
   - 水源候補 × 取水口候補 × 発電所候補
   - 総組み合わせ数 = N_水源 × N_取水口 × N_発電所
   - 例：40 × 40 × 40 = 64,000通り

2. **物理的制約のチェック**（各組み合わせについて）
   ```
   # 標高の順序チェック
   if 水源標高 <= 取水口標高:
       この組み合わせは無効 → スキップ
   if 取水口標高 <= 発電所標高:
       この組み合わせは無効 → スキップ
   
   # 有効落差の計算
   有効落差 = 水源標高 - 発電所標高
   
   if 有効落差 <= 0:
       この組み合わせは無効 → スキップ
   ```

3. **発電量の計算**（有効な組み合わせについて）
   ```
   # 基本パラメータ
   重力加速度 g = 9.8 m/s²
   流量 Q = 水源の推定流量 (m³/s)
   有効落差 H = 水源標高 - 発電所標高 (m)
   
   # 理論水力 (kW)
   理論水力 = g × Q × H / 1000
   
   # 損失の考慮
   管路損失係数 = 0.90  # 管路摩擦、曲がりなどで10%損失
   水車効率 = 0.85      # ペルトン水車などで85%程度
   発電機効率 = 0.95    # 発電機で95%程度
   
   総合効率 = 管路損失係数 × 水車効率 × 発電機効率
            = 0.90 × 0.85 × 0.95
            = 0.727 ≈ 0.73
   
   # 簡易計算では0.8を使用
   簡易効率 = 0.80
   
   # 実発電量 (kW)
   発電量 = 理論水力 × 簡易効率
          = 9.8 × Q × H × 0.8 / 1000
          = 0.00784 × Q × H  (kW)
   ```

4. **距離の計算**（参考情報として）
   ```
   # 緯度経度から距離を計算（Haversine公式）
   水源-取水口間距離 = haversine(水源座標, 取水口座標)
   取水口-発電所間距離 = haversine(取水口座標, 発電所座標)
   水源-発電所間直線距離 = haversine(水源座標, 発電所座標)
   ```

5. **組み合わせデータの記録**
   ```
   各組み合わせについて以下を記録：
   - 水源の情報（座標、標高、流量）
   - 取水口の情報（座標、標高）
   - 発電所の情報（座標、標高）
   - 有効落差 (m)
   - 発電量 (kW)
   - 各施設間の距離 (km)
   ```

**出力データ**:
- 有効な組み合わせリスト（発電量 > 0のもの）
- 評価済み組み合わせ数：○○通り

---

### ステップ10：ランキングと最適解の選出

**目的**: 発電量の高い組み合わせを選出

**手順**:
1. **ソート**
   - すべての有効な組み合わせを発電量で降順ソート
   - 最大発電量の組み合わせが第1位に

2. **上位候補の選出**
   - 上位M個（デフォルト30個）を最終候補として選出
   - 選出理由：
     - 上位候補は複数の選択肢を提供
     - 現地条件（地権、環境など）で選択可能

3. **結果の整理**
   ```
   各候補について：
   - 順位 (1位、2位、...)
   - 発電量 (kW)
   - 有効落差 (m)
   - 水源・取水口・発電所の詳細座標と標高
   - 施設間距離
   ```

**出力データ**:
- 最終候補リスト：上位M個の組み合わせ
- 第1位の発電量：最大○○ kW
- 平均発電量：○○ kW

---

### ステップ11：可視化とファイル出力

**目的**: 結果を分かりやすく表示・保存

**可視化**:
1. **地図の生成**
   - Foliumライブラリで対話型地図を作成
   - 上位3組の候補地をプロット
     - 赤・青・緑でマーカーを色分け
     - 施設間を線で接続
   - 全グリッドポイントを標高で色分け表示（背景）

2. **グラフの生成**
   - 標高分布図：グリッドポイントの標高を散布図で表示
   - 発電量比較図：上位候補の発電量を棒グラフで比較
   - 有効落差比較図：上位候補の落差を棒グラフで比較
   - 標高プロファイル図：上位3組の施設配置を線グラフで表示

**ファイル出力**:
1. **HTMLファイル**：対話型地図を保存
2. **CSVファイル**：全候補の詳細データを保存
3. **PNGファイル**：各グラフを画像として保存
4. **テキストファイル**：分析結果のサマリーを保存

**出力データ**:
- ファイル保存先：`deta/<日時>/`
- ファイル数：通常7個（HTML×1、CSV×1、PNG×4、TXT×1）

---

## 算出結果の例

### 長野県の場合

**入力**:
- 地名：「長野県」
- グリッドサイズ：28×28（自動設定）
- 候補数：各40箇所（自動設定）

**ステップ別の出力**:
1. **範囲確定**：面積 13,509 km²、境界頂点数 36,137個
2. **グリッド生成**：784点 → 378点（境界内）
3. **標高取得**：325m ～ 2,857m、平均 1,149m
4. **河川取得**：26,193本（river: 5,128本、stream: 21,065本）
5. **勾配計算**：5.0 ～ 1,998.0
6. **水源選定**：40箇所、平均標高 2,003m、平均流量 0.45 m³/s
7. **取水口選定**：40箇所、平均標高 1,517m
8. **発電所選定**：40箇所、平均標高 488m
9. **組み合わせ評価**：64,000通り評価
10. **最適解選出**：上位30組、最大発電量 1,146 kW

**実行時間**：約97秒

---

## 出力されるファイル

結果は `deta/日時/` フォルダに保存されます：

| ファイル名 | 内容 |
|----------|------|
| `hydro_map_top3_<地名>_<日時>.html` | 上位3つの候補地を表示した地図（英語表記） |
| `hydro_sites_<地名>_<日時>.csv` | すべての候補地の詳細データ（英語列名） |
| `1_Elevation_Distribution_<地名>_<日時>.png` | 標高分布グラフ |
| `2_Power_Output_Comparison_<地名>_<日時>.png` | 発電量比較グラフ |
| `3_Effective_Head_Comparison_<地名>_<日時>.png` | 有効落差比較グラフ |
| `4_Facility_Elevation_Profile_<地名>_<日時>.png` | 施設配置プロファイル |
| `summary_<地名>_<日時>.txt` | 分析結果のサマリー（日本語） |

※ ファイル名の地名部分は日本語、グラフのタイトルやラベルは英語で保存されます

## 詳細設定（任意）

もっと細かく調整したい場合：

```python
selector.run_analysis(
    grid_size=30,           # 探索グリッドの密度（大きいほど詳細、計算時間増）
    candidates_per_type=40, # 各タイプの候補数（多いほど選択肢が増える）
    top_n=30               # 最終的に表示する候補数
)
```

※ 通常は自動で最適な値が設定されるので、指定しなくてOKです

**自動設定のロジック**:
- `grid_size`: 面積に応じて自動計算（目標密度: 約15 km²/点）
- `candidates_per_type`: グリッド点数の5%（最小20、最大80）
- `top_n`: 組合せ総数に応じて10～30を自動選択

## 結果の見方

### 地図の見方
- **マーカーの色**で候補地の順位が分かります
  - 赤：第1位、青：第2位、緑：第3位
- **線**で施設同士がつながっています（水の流れる経路）
- マーカーをクリックすると詳細情報が表示されます
  - 水源：標高、推定流量
  - 取水口：標高
  - 発電所：標高、発電量

### CSVデータの列
- `Rank`: 順位
- `Power_kW`: 推定発電量（キロワット）
- `Effective_Head_m`: 有効落差（メートル）
- `WS_Lat`, `WS_Lon`: 水源の緯度・経度
- `WS_Elevation_m`: 水源の標高
- `WS_Flow_m3_s`: 水源の推定流量（m³/s）
- `Intake_Lat`, `Intake_Lon`: 取水口の緯度・経度
- `Intake_Elevation_m`: 取水口の標高
- `PH_Lat`, `PH_Lon`: 発電所の緯度・経度
- `PH_Elevation_m`: 発電所の標高
- `WS_Intake_Distance_km`: 水源-取水口間の距離
- `Intake_PH_Distance_km`: 取水口-発電所間の距離
- `WS_PH_Distance_km`: 水源-発電所間の直線距離

## 注意事項

- このシステムの結果は**参考値**です
- 実際の建設には、詳細な現地調査や許可申請が必要です
- 計算時間は地域の広さによって変わります
  - 小規模な市町村：数分程度
  - 県レベル：数十分～1時間程度
- インターネット接続が必要です（地図や標高データを取得するため）
- 河川流量は推定値であり、実際の測定値とは異なる場合があります

## トラブルシューティング

### エラーが出た場合
1. **地名が見つからない**: 正式な市区町村名・都道府県名を入力してください
2. **タイムアウトエラー**: インターネット接続を確認してください
3. **計算が終わらない**: grid_sizeを小さくしてみてください（例: 20）

### 実行が遅い場合
```python
# 高速モード（精度は少し下がります）
selector.run_analysis(grid_size=15, candidates_per_type=20, top_n=10)
```

## 技術的な詳細

### 使用API
- **Nominatim API**: 地名から座標と行政区画境界を取得
- **Open-Elevation API**: 標高データの一括取得
- **Overpass API**: OpenStreetMapから河川データを取得

### データ精度
- **境界精度**: 行政区画の面積誤差は通常1%未満
- **標高精度**: ±数メートル程度（地形データの解像度に依存）
- **流量推定**: 河川タイプと距離から推定（実測値ではない）

### 計算の最適化
- グリッドポイントと河川データを境界内にフィルタリング
- 標高データはバッチ処理で高速取得（150点/リクエスト）
- 組合せ評価は全探索（数万～数十万通り）で最適解を保証


<a href="https://colab.research.google.com/github/tsuka22120/4IE2/blob/main/ED.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 💧 水力発電所候補地選定システム

---

## 🎯 このシステムでできること

地名を入力するだけで、その地域に**水力発電所を建設できそうな場所**を自動で探し出します。

```
[入力] 地名を入力 -> [分析] 地形を分析 -> [出力] 最適な発電所候補地を提案
```

---

## [FLOW] システムの流れ

```
┌─────────────────────────────────────────────────────────────────┐
│                                                                 │
│   STEP 1: 地域を指定                                            │
│   ┌─────────────────┐                                           │
│   │  例: "松本市"   │  ← 調べたい地域名を入力                   │
│   └────────┬────────┘                                           │
│            ▼                                                    │
│   STEP 2: 地形データ取得                                        │
│   ┌─────────────────┐                                           │
│   │ [山] 標高データ   │  ← どこが高い/低いかを調査               │
│   │ [川] 河川データ   │  ← 川がどこを流れているかを調査          │
│   └────────┬────────┘                                           │
│            ▼                                                    │
│   STEP 3: 候補地を選定                                          │
│   ┌─────────────────┐                                           │
│   │ [水源] 水源         │  ← 水を取る場所（高い所）                 │
│   │ [取水] 取水口       │  ← 水を集める場所（中間）                 │
│   │ [発電] 発電所       │  ← 発電する場所（低い所）                 │
│   └────────┬────────┘                                           │
│            ▼                                                    │
│   STEP 4: 発電量を計算                                          │
│   ┌─────────────────────────────────────────┐                   │
│   │  発電量 = 水の落差 × 水の量 × 効率      │                   │
│   │  （高低差が大きいほど、たくさん発電！） │                   │
│   └────────┬────────────────────────────────┘                   │
│            ▼                                                    │
│   STEP 5: 結果を出力                                            │
│   ┌─────────────────┐                                           │
│   │ [MAP] 地図で表示   │  ← 候補地の位置がひと目でわかる          │
│   │ [GRAPH] グラフ表示   │  ← 発電量を比較できる                     │
│   │ [FILE] ファイル保存 │  ← 結果を後で見返せる                     │
│   └─────────────────┘                                           │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

---

## [FOLDER] 出力されるファイル

実行するたびに `deta/yyyymmddhhmm/` フォルダに結果が保存されます。

| ファイル | 内容 | 見方 |
|:---------|:-----|:-----|
| [MAP] `hydro_map_XX.html` | 候補地マップ | ブラウザで開くと地図が見れる |
| [DATA] `hydro_sites_XX.csv` | 候補地データ | Excelで開ける数値データ |
| [TEXT] `summary_XX.txt` | 結果まとめ | メモ帳で開けるサマリー |
| [GRAPH] `1_Elevation_XX.png` | 標高分布図 | 地形の高低がわかる |
| [GRAPH] `2_Power_XX.png` | 発電量比較 | どれが一番発電できるか |
| [GRAPH] `3_Effective_XX.png` | 落差比較 | 高低差の比較 |
| [GRAPH] `4_Facility_XX.png` | 施設配置図 | 3施設の標高関係 |

---

## [START] 使い方（3ステップ）

### ステップ1: 設定を確認
下のメインセルで `LOCATION_NAME = "松本市"` の部分を、調べたい地域名に変更

### ステップ2: 実行
メインセル（一番大きいコードセル）を実行 -> 数分待つ

### ステップ3: 結果を確認
- [MAP] 地図とグラフがノートブック内に表示される
- [FILE] `deta/` フォルダに結果ファイルが保存される

---

## [TOOL] 便利な機能

このノートブックには、過去の結果を確認できる機能もあります：

```python
# 過去の結果一覧を見る
list_output_directories(show_files=True)

# 特定の結果を詳しく見る
explore_output_files("202601130409")

# 過去の結果を読み込む
results = load_results_from_directory("202601130409")
```

---

In [ ]:
# ============================================================
# detaフォルダ内ファイル探索・管理機能
# ============================================================
import os
import sys
from datetime import datetime
import pandas as pd

def is_colab_environment():
    """Google Colab環境かどうかを判定"""
    return 'google.colab' in sys.modules

def get_deta_base_path():
    """
    detaフォルダのベースパスを取得
    
    Colab環境: /content/deta
    ローカル環境: カレントディレクトリまたはノートブックディレクトリの deta/
    """
    # Colab環境の場合
    if is_colab_environment():
        colab_deta = "/content/deta"
        if not os.path.exists(colab_deta):
            os.makedirs(colab_deta, exist_ok=True)
        return colab_deta
    
    # ローカル環境の場合
    current_dir = os.getcwd()
    
    # いくつかの候補パスをチェック
    candidates = [
        os.path.join(current_dir, "deta"),
        os.path.join(current_dir, "ED", "deta"),
        os.path.join(os.path.dirname(current_dir), "deta"),
    ]
    
    for path in candidates:
        if os.path.exists(path):
            return path
    
    # 存在しない場合はデフォルトで作成
    default_path = os.path.join(current_dir, "deta")
    os.makedirs(default_path, exist_ok=True)
    return default_path

def list_output_directories(show_files=False, max_dirs=None):
    """
    detaフォルダ内のすべての実行結果ディレクトリを一覧表示
    
    Parameters:
    -----------
    show_files : bool
        各ディレクトリ内のファイルも表示するかどうか
    max_dirs : int or None
        表示する最大ディレクトリ数（新しい順）
    
    Returns:
    --------
    list : ディレクトリ情報のリスト
    """
    deta_path = get_deta_base_path()
    
    print(f"\n{'='*70}")
    print(f"[FOLDER] detaフォルダ内のディレクトリ一覧")
    print(f"{'='*70}")
    if is_colab_environment():
        print(f"[!] Colab環境で実行中 - ファイルは /content/deta に保存されます")
    print(f"パス: {deta_path}\n")
    
    if not os.path.exists(deta_path):
        print("[X] detaフォルダが存在しません")
        return []
    
    # ディレクトリ一覧を取得（yyyymmddhhmm形式のみ）
    directories = []
    for item in os.listdir(deta_path):
        item_path = os.path.join(deta_path, item)
        if os.path.isdir(item_path) and item.isdigit() and len(item) == 12:
            # ディレクトリ内のファイル数をカウント
            files = [f for f in os.listdir(item_path) if os.path.isfile(os.path.join(item_path, f))]
            
            # タイムスタンプを解析
            try:
                dt = datetime.strptime(item, "%Y%m%d%H%M")
                dt_str = dt.strftime("%Y年%m月%d日 %H:%M")
            except:
                dt_str = item
            
            directories.append({
                'name': item,
                'datetime': dt_str,
                'path': item_path,
                'file_count': len(files),
                'files': files
            })
    
    # 新しい順にソート
    directories.sort(key=lambda x: x['name'], reverse=True)
    
    if max_dirs:
        directories = directories[:max_dirs]
    
    # 表示
    print(f"総ディレクトリ数: {len(directories)}\n")
    
    for i, d in enumerate(directories, 1):
        print(f"[DIR] [{i:02d}] {d['name']} ({d['datetime']}) - {d['file_count']}ファイル")
        
        if show_files and d['files']:
            for f in sorted(d['files']):
                # ファイルタイプを判定
                ext = os.path.splitext(f)[1].lower()
                if ext == '.html':
                    icon = "[MAP] "
                elif ext == '.csv':
                    icon = "[DATA]"
                elif ext == '.png':
                    icon = "[IMG]"
                elif ext == '.txt':
                    icon = "[TXT]"
                else:
                    icon = "[FILE]"
                print(f"      {icon} {f}")
            print()
    
    print(f"\n{'='*70}")
    return directories


def explore_output_files(dirname=None, show_details=True):
    """
    特定のディレクトリ内のファイルを詳細に探索
    
    Parameters:
    -----------
    dirname : str or None
        探索するディレクトリ名（Noneの場合は最新のディレクトリ）
    show_details : bool
        ファイルサイズなどの詳細を表示するか
    
    Returns:
    --------
    dict : ファイル情報の辞書
    """
    deta_path = get_deta_base_path()
    
    # ディレクトリ名が指定されていない場合は最新を使用
    if dirname is None:
        dirs = sorted([d for d in os.listdir(deta_path) 
                      if os.path.isdir(os.path.join(deta_path, d)) and d.isdigit() and len(d) == 12],
                     reverse=True)
        if not dirs:
            print("[X] ディレクトリが見つかりません")
            return {}
        dirname = dirs[0]
    
    dirpath = os.path.join(deta_path, dirname)
    
    if not os.path.exists(dirpath):
        print(f"[X] ディレクトリが見つかりません: {dirname}")
        return {}
    
    print(f"\n{'='*70}")
    print(f"[DIR] ディレクトリ詳細: {dirname}")
    print(f"{'='*70}")
    print(f"パス: {dirpath}\n")
    
    # ファイル情報を収集
    file_info = {
        'maps': [],
        'data': [],
        'graphs': [],
        'summary': [],
        'logs': [],
        'others': []
    }
    
    for filename in os.listdir(dirpath):
        filepath = os.path.join(dirpath, filename)
        if not os.path.isfile(filepath):
            continue
        
        # ファイルサイズ取得
        size = os.path.getsize(filepath)
        if size < 1024:
            size_str = f"{size} B"
        elif size < 1024 * 1024:
            size_str = f"{size / 1024:.1f} KB"
        else:
            size_str = f"{size / (1024 * 1024):.1f} MB"
        
        file_entry = {
            'name': filename,
            'path': filepath,
            'size': size,
            'size_str': size_str
        }
        
        # ファイルタイプ別に分類
        ext = os.path.splitext(filename)[1].lower()
        if ext == '.html':
            file_info['maps'].append(file_entry)
        elif ext == '.csv':
            file_info['data'].append(file_entry)
        elif ext == '.png':
            file_info['graphs'].append(file_entry)
        elif ext == '.txt':
            if 'summary' in filename.lower():
                file_info['summary'].append(file_entry)
            elif 'log' in filename.lower():
                file_info['logs'].append(file_entry)
            else:
                file_info['others'].append(file_entry)
        else:
            file_info['others'].append(file_entry)
    
    # 表示
    categories = [
        ('[MAP] 地図ファイル (HTML)', 'maps'),
        ('[DATA] データファイル (CSV)', 'data'),
        ('[IMG] グラフファイル (PNG)', 'graphs'),
        ('[TXT] サマリーファイル', 'summary'),
        ('[LOG] ログファイル', 'logs'),
        ('[FILE] その他', 'others')
    ]
    
    total_files = 0
    for title, key in categories:
        files = file_info[key]
        if files:
            print(f"\n{title}: {len(files)}件")
            print("-" * 50)
            for f in sorted(files, key=lambda x: x['name']):
                print(f"  • {f['name']} ({f['size_str']})")
                total_files += 1
    
    print(f"\n{'='*70}")
    print(f"総ファイル数: {total_files}")
    print(f"{'='*70}")
    
    return file_info


def get_all_output_summary():
    """
    全ディレクトリの出力サマリーを取得してDataFrameで返す
    
    Returns:
    --------
    pandas.DataFrame : 全ディレクトリのサマリー情報
    """
    deta_path = get_deta_base_path()
    
    print(f"\n{'='*70}")
    print(f"[SUMMARY] 全実行結果サマリー")
    print(f"{'='*70}\n")
    
    summary_data = []
    
    for dirname in os.listdir(deta_path):
        dirpath = os.path.join(deta_path, dirname)
        if not os.path.isdir(dirpath) or not dirname.isdigit() or len(dirname) != 12:
            continue
        
        # ファイル情報を収集
        files = os.listdir(dirpath)
        
        # 地域名を抽出（ファイル名から）
        location = "不明"
        for f in files:
            if f.startswith('hydro_map_') and f.endswith('.html'):
                # hydro_map_{地域名}_{タイムスタンプ}.html から地域名を抽出
                parts = f.replace('hydro_map_', '').replace('.html', '').rsplit('_', 1)
                if len(parts) >= 1:
                    location = parts[0]
                    break
            elif f.startswith('hydro_sites_') and f.endswith('.csv'):
                parts = f.replace('hydro_sites_', '').replace('.csv', '').rsplit('_', 1)
                if len(parts) >= 1:
                    location = parts[0]
                    break
        
        # ファイルタイプ別カウント
        html_count = len([f for f in files if f.endswith('.html')])
        csv_count = len([f for f in files if f.endswith('.csv')])
        png_count = len([f for f in files if f.endswith('.png')])
        txt_count = len([f for f in files if f.endswith('.txt')])
        
        try:
            dt = datetime.strptime(dirname, "%Y%m%d%H%M")
            dt_str = dt.strftime("%Y-%m-%d %H:%M")
        except:
            dt_str = dirname
        
        summary_data.append({
            'ディレクトリ': dirname,
            '日時': dt_str,
            '地域名': location,
            'HTML': html_count,
            'CSV': csv_count,
            'PNG': png_count,
            'TXT': txt_count,
            '総ファイル数': len(files)
        })
    
    # DataFrameに変換してソート
    df = pd.DataFrame(summary_data)
    if not df.empty:
        df = df.sort_values('ディレクトリ', ascending=False).reset_index(drop=True)
        print(df.to_string(index=False))
    else:
        print("[X] 実行結果が見つかりません")
    
    return df


def load_results_from_directory(dirname=None):
    """
    特定のディレクトリから結果を読み込んで表示
    
    Parameters:
    -----------
    dirname : str or None
        読み込むディレクトリ名（Noneの場合は最新のディレクトリ）
    
    Returns:
    --------
    dict : 読み込んだデータを含む辞書
    """
    deta_path = get_deta_base_path()
    
    # ディレクトリ名が指定されていない場合は最新を使用
    if dirname is None:
        dirs = sorted([d for d in os.listdir(deta_path) 
                      if os.path.isdir(os.path.join(deta_path, d)) and d.isdigit() and len(d) == 12],
                     reverse=True)
        if not dirs:
            print("[X] ディレクトリが見つかりません")
            return {}
        dirname = dirs[0]
    
    dirpath = os.path.join(deta_path, dirname)
    
    if not os.path.exists(dirpath):
        print(f"[X] ディレクトリが見つかりません: {dirname}")
        return {}
    
    print(f"\n{'='*70}")
    print(f"[LOAD] 結果読み込み: {dirname}")
    print(f"{'='*70}\n")
    
    results = {}
    
    # CSVファイルを読み込み
    for f in os.listdir(dirpath):
        if f.endswith('.csv') and 'hydro_sites' in f:
            csv_path = os.path.join(dirpath, f)
            df = pd.read_csv(csv_path)
            results['dataframe'] = df
            print(f"✓ CSVデータ読み込み: {f} ({len(df)}行)")
            
            # データのサマリーを表示
            print(f"\n[データサマリー]")
            if 'Power_kW' in df.columns:
                print(f"  発電量範囲: {df['Power_kW'].min():.1f} ~ {df['Power_kW'].max():.1f} kW")
            if 'Effective_Head_m' in df.columns:
                print(f"  落差範囲: {df['Effective_Head_m'].min():.1f} ~ {df['Effective_Head_m'].max():.1f} m")
            break
    
    # サマリーファイルを読み込み
    for f in os.listdir(dirpath):
        if f.endswith('.txt') and 'summary' in f:
            summary_path = os.path.join(dirpath, f)
            with open(summary_path, 'r', encoding='utf-8-sig') as file:
                summary_content = file.read()
            results['summary'] = summary_content
            print(f"\n✓ サマリー読み込み: {f}")
            print("\n" + summary_content[:500] + "..." if len(summary_content) > 500 else "\n" + summary_content)
            break
    
    return results


def compare_results(dirname1, dirname2):
    """
    2つのディレクトリの結果を比較
    
    Parameters:
    -----------
    dirname1 : str
        比較元のディレクトリ名
    dirname2 : str
        比較先のディレクトリ名
    
    Returns:
    --------
    dict : 比較結果
    """
    deta_path = get_deta_base_path()
    
    print(f"\n{'='*70}")
    print(f"[COMPARE] 結果比較: {dirname1} vs {dirname2}")
    print(f"{'='*70}\n")
    
    # 両方のCSVを読み込み
    csv1_path = None
    csv2_path = None
    
    for f in os.listdir(os.path.join(deta_path, dirname1)):
        if f.endswith('.csv') and 'hydro_sites' in f:
            csv1_path = os.path.join(deta_path, dirname1, f)
            break
    
    for f in os.listdir(os.path.join(deta_path, dirname2)):
        if f.endswith('.csv') and 'hydro_sites' in f:
            csv2_path = os.path.join(deta_path, dirname2, f)
            break
    
    if csv1_path is None or csv2_path is None:
        print("[X] CSVファイルが見つかりません")
        return {}
    
    df1 = pd.read_csv(csv1_path)
    df2 = pd.read_csv(csv2_path)
    
    print(f"[DATE] {dirname1}: {len(df1)}件の候補地")
    print(f"[DATE] {dirname2}: {len(df2)}件の候補地")
    
    # 発電量の比較
    if 'Power_kW' in df1.columns and 'Power_kW' in df2.columns:
        print(f"\n[POWER] 発電量比較:")
        print(f"  {dirname1} - 最大: {df1['Power_kW'].max():.1f} kW, 平均: {df1['Power_kW'].mean():.1f} kW")
        print(f"  {dirname2} - 最大: {df2['Power_kW'].max():.1f} kW, 平均: {df2['Power_kW'].mean():.1f} kW")
    
    # 落差の比較
    if 'Effective_Head_m' in df1.columns and 'Effective_Head_m' in df2.columns:
        print(f"\n[HEAD] 落差比較:")
        print(f"  {dirname1} - 最大: {df1['Effective_Head_m'].max():.1f} m, 平均: {df1['Effective_Head_m'].mean():.1f} m")
        print(f"  {dirname2} - 最大: {df2['Effective_Head_m'].max():.1f} m, 平均: {df2['Effective_Head_m'].mean():.1f} m")
    
    return {'df1': df1, 'df2': df2}


# ============================================================
# 使用例の表示
# ============================================================
print("[OK] ファイル探索機能が読み込まれました")
print("-" * 50)
if is_colab_environment():
    print("[!] Google Colab環境で実行中")
    print("   ファイルは /content/deta/ に保存されます")
    print("-" * 50)
print("利用可能な関数:")
print("  • list_output_directories(show_files=False, max_dirs=None)")
print("    → ディレクトリ一覧を表示")
print("  • explore_output_files(dirname=None, show_details=True)")
print("    → 特定ディレクトリのファイルを詳細表示")
print("  • get_all_output_summary()")
print("    → 全ディレクトリのサマリーをDataFrameで取得")
print("  • load_results_from_directory(dirname=None)")
print("    → 特定ディレクトリの結果を読み込んで表示")
print("  • compare_results(dirname1, dirname2)")
print("    → 2つのディレクトリの結果を比較")
print("-" * 50)

📁 ファイル探索機能が読み込まれました
--------------------------------------------------
利用可能な関数:
  • list_output_directories(show_files=False, max_dirs=None)
    → ディレクトリ一覧を表示
  • explore_output_files(dirname=None, show_details=True)
    → 特定ディレクトリのファイルを詳細表示
  • get_all_output_summary()
    → 全ディレクトリのサマリーをDataFrameで取得
  • load_results_from_directory(dirname=None)
    → 特定ディレクトリの結果を読み込んで表示
  • compare_results(dirname1, dirname2)
    → 2つのディレクトリの結果を比較
--------------------------------------------------


In [ ]:
# ============================================================
# 水力発電候補地選定システム - 統合版
# ============================================================

# ============================================================
# 設定値・定数定義（拡張性のため上部に配置）
# ============================================================

# 地域設定
LOCATION_NAME = "松本市"  # 対象地域名（変更可能）

# グリッドとサンプリング設定
GRID_SIZE = 20                    # グリッドサイズ（自動計算する場合はNone）
CANDIDATES_PER_TYPE = 20          # 各タイプの候補数（自動計算する場合はNone）
TOP_N = 5                         # 出力する上位組合せ数（自動計算する場合はNone）

# API設定
ELEVATION_BATCH_SIZE = 100        # 標高API一括取得サイズ
ELEVATION_RETRY = 2               # 標高取得リトライ回数
RIVER_QUERY_TIMEOUT = 60          # 河川データ取得タイムアウト（秒）

# 物理定数
GRAVITY = 9.8                     # 重力加速度 (m/s²)
WATER_DENSITY = 1000              # 水の密度 (kg/m³)
TURBINE_EFFICIENCY = 0.8          # タービン効率

# 推定パラメータ
RIVER_WIDTH_ESTIMATES = {         # 河川タイプ別の幅推定値 (m)
    'river': 20.0,
    'stream': 5.0,
    'canal': 10.0
}
DEFAULT_RIVER_FLOW = 3.0          # デフォルト河川流量 (m³/s)
RIVER_FLOW_DECAY_DISTANCE = 5     # 流量減衰距離パラメータ (km)
RIVER_MAX_DISTANCE = 15           # 河川有効範囲 (km)
FLOW_LOSS_DISTANCE = 15           # 水路損失距離パラメータ (km)
DISTANCE_PENALTY_FACTOR = 20      # 距離ペナルティ係数 (km)

# スコアリング重み
WATER_SOURCE_WEIGHTS = {          # 水源スコアリング重み
    'elevation': 0.6,
    'slope': 0.4
}
INTAKE_WEIGHTS = {                # 取水口スコアリング重み
    'middle_elevation': 0.6,
    'gentle_slope': 0.4
}
POWERHOUSE_WEIGHTS = {            # 発電所スコアリング重み
    'low_elevation': 0.7,
    'gentle_slope': 0.3
}

# 可視化設定
DISPLAY_TOP_N_ON_MAP = 3          # 地図上に表示する上位N組
GRID_SAMPLE_MAX = 200             # 地図上のグリッドポイント最大表示数
MAP_COLORS = ['red', 'blue', 'green', 'purple', 'orange']  # 組合せの色
GRAPH_DPI = 150                   # グラフ解像度

# 自動計算パラメータ
AUTO_GRID_DENSITY = 15            # 自動グリッド密度 (km²/点)
AUTO_GRID_MIN = 400               # 自動グリッド最小値
AUTO_GRID_MAX = 10000             # 自動グリッド最大点数
AUTO_CANDIDATES_RATIO = 0.05      # 自動候補数比率（グリッドの5%）
AUTO_CANDIDATES_MIN = 20          # 自動候補数最小値
AUTO_CANDIDATES_MAX = 80          # 自動候補数最大値

# ============================================================
# ライブラリのインポート
# ============================================================
import folium
from folium import plugins
import numpy as np
import pandas as pd
import requests
from geopy.geocoders import Nominatim
from geopy.distance import geodesic
from scipy.ndimage import gaussian_filter
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import time
from datetime import datetime
from itertools import combinations
import warnings
from shapely.geometry import Point, Polygon, MultiPolygon, LineString
import sys
import os
from io import StringIO
import threading
import matplotlib

warnings.filterwarnings('ignore')

# ============================================================
# 日本語フォント設定（matplotlib）
# ============================================================
def setup_japanese_font():
    """matplotlibで日本語フォントを設定"""
    import platform
    system = platform.system()
    
    if system == 'Windows':
        # Windows: MS Gothic, Yu Gothic, Meiryo など
        font_candidates = ['MS Gothic', 'Yu Gothic', 'Meiryo', 'MS Mincho']
    elif system == 'Darwin':  # macOS
        font_candidates = ['Hiragino Sans', 'Hiragino Kaku Gothic ProN', 'AppleGothic']
    else:  # Linux
        font_candidates = ['IPAGothic', 'IPAPGothic', 'Noto Sans CJK JP', 'TakaoGothic']
    
    # 利用可能なフォントを検出
    from matplotlib.font_manager import fontManager
    available_fonts = [f.name for f in fontManager.ttflist]
    
    for font in font_candidates:
        if font in available_fonts:
            plt.rcParams['font.family'] = font
            print(f"✓ 日本語フォント設定: {font}")
            return font
    
    # フォールバック: sans-serifを使用
    plt.rcParams['font.family'] = 'sans-serif'
    if system == 'Windows':
        plt.rcParams['font.sans-serif'] = ['MS Gothic', 'Yu Gothic', 'Meiryo'] + plt.rcParams['font.sans-serif']
    elif system == 'Darwin':
        plt.rcParams['font.sans-serif'] = ['Hiragino Sans', 'Hiragino Kaku Gothic ProN'] + plt.rcParams['font.sans-serif']
    else:
        plt.rcParams['font.sans-serif'] = ['IPAGothic', 'Noto Sans CJK JP'] + plt.rcParams['font.sans-serif']
    
    print(f"⚠️ 日本語フォント: フォールバック設定を使用")
    return None

# matplotlibのマイナス記号対策
plt.rcParams['axes.unicode_minus'] = False
setup_japanese_font()

# ============================================================
# ログ機能
# ============================================================
class TeeOutput:
    """標準出力とファイルの両方に出力するクラス"""
    def __init__(self, file_path):
        self.terminal = sys.stdout
        self.log_file = open(file_path, 'w', encoding='utf-8')
        
    def write(self, message):
        self.terminal.write(message)
        self.log_file.write(message)
        self.log_file.flush()
        
    def flush(self):
        self.terminal.flush()
        self.log_file.flush()
        
    def close(self):
        if self.log_file and not self.log_file.closed:
            self.log_file.close()

_tee_output = None

def start_logging(location_name="default"):
    """ログ記録を開始"""
    global _tee_output
    
    if _tee_output:
        stop_logging()
    
    timestamp = datetime.now().strftime("%Y%m%d%H%M")
    log_dir = f"deta/{timestamp}"
    os.makedirs(log_dir, exist_ok=True)
    
    log_filename = f"execution_log_{location_name}_{timestamp}.txt"
    log_path = os.path.join(log_dir, log_filename)
    
    _tee_output = TeeOutput(log_path)
    sys.stdout = _tee_output
    
    print(f"{'='*70}")
    print(f"水力発電候補地選定システム - 実行ログ")
    print(f"{'='*70}")
    print(f"地域名: {location_name}")
    print(f"開始時刻: {datetime.now().strftime('%Y年%m月%d日 %H:%M:%S')}")
    print(f"ログファイル: {log_path}")
    print(f"{'='*70}\n")

def stop_logging():
    """ログ記録を終了"""
    global _tee_output
    
    if _tee_output:
        print(f"\n{'='*70}")
        print(f"終了時刻: {datetime.now().strftime('%Y年%m月%d日 %H:%M:%S')}")
        print(f"{'='*70}")
        
        sys.stdout = _tee_output.terminal
        _tee_output.close()
        _tee_output = None

# ============================================================
# メインクラス: HydroSiteSelector
# ============================================================
class HydroSiteSelector:
    """水力発電候補地選定クラス"""
    
    def __init__(self, location_name):
        self.location_name = location_name
        self.center_lat = None
        self.center_lon = None
        self.boundary_coords = None
        self.boundary_polygon = None
        self.grid_points = []
        self.elevation_data = None
        self.river_data = []
        self.river_flow_estimates = {}
        self.candidates = {'water_sources': [], 'intakes': [], 'powerhouses': []}
        self.best_combinations = []
        self.area_km2 = 0
        self._status = {'stage': 'init', 'progress': 0, 'message': '初期化完了'}
    
    def update_status(self, stage=None, progress=None, message=None):
        """進捗状況を更新"""
        if stage:
            self._status['stage'] = stage
        if progress is not None:
            self._status['progress'] = progress
        if message:
            self._status['message'] = message
    
    def get_status(self):
        """現在の進捗状況を取得"""
        return self._status.copy()
    
    def get_location_coordinates(self):
        """地名から座標と境界を取得"""
        print(f"地域情報取得中: {self.location_name}")
        self.update_status(stage='geocode', progress=5, message='座標取得中')
        
        geolocator = Nominatim(user_agent="hydro_power_demo_v3")
        
        try:
            location = geolocator.geocode(self.location_name, timeout=10)
            if not location:
                print(f"エラー: '{self.location_name}' の座標が見つかりません")
                return False
            
            self.center_lat = location.latitude
            self.center_lon = location.longitude
            print(f"[OK] 中心座標: ({self.center_lat:.4f}, {self.center_lon:.4f})")
            
            time.sleep(1)
            
            location_with_geom = geolocator.geocode(
                self.location_name,
                geometry='geojson',
                timeout=10
            )
            
            if location_with_geom and hasattr(location_with_geom, 'raw'):
                geojson = location_with_geom.raw.get('geojson', {})
                geom_type = geojson.get('type', '')
                coords = geojson.get('coordinates', [])
                
                if geom_type == 'Polygon' and coords:
                    exterior = coords[0]
                    self.boundary_coords = [(lat, lon) for lon, lat in exterior]
                    self.boundary_polygon = Polygon([(lon, lat) for lat, lon in self.boundary_coords])
                    self.area_km2 = self._calculate_area()
                    print(f"[OK] 行政区画境界取得: {len(self.boundary_coords)}点")
                    print(f"[OK] 概算面積: {self.area_km2:.1f} km²")
                    
                elif geom_type == 'MultiPolygon' and coords:
                    all_coords = []
                    polygons = []
                    for polygon_coords in coords:
                        exterior = polygon_coords[0]
                        poly_points = [(lat, lon) for lon, lat in exterior]
                        all_coords.extend(poly_points)
                        polygons.append(Polygon([(lon, lat) for lat, lon in poly_points]))
                    
                    self.boundary_coords = all_coords
                    self.boundary_polygon = MultiPolygon(polygons)
                    self.area_km2 = self._calculate_area()
                    print(f"[OK] 複合行政区画境界取得: {len(coords)}ポリゴン, 計{len(self.boundary_coords)}点")
                    print(f"[OK] 概算面積: {self.area_km2:.1f} km²")
                else:
                    self._create_default_boundary()
            else:
                self._create_default_boundary()
            
            self.update_status(stage='geocode', progress=10, message='座標取得完了')
            return True
            
        except Exception as e:
            print(f"座標取得エラー: {e}")
            return False
    
    def _create_default_boundary(self, radius_km=5):
        """デフォルトの円形境界を作成"""
        print(f"  境界情報なし → 半径{radius_km}kmの探索範囲を設定")
        
        deg_per_km = 1 / 111
        n_points = 36
        self.boundary_coords = []
        
        for i in range(n_points):
            angle = 2 * np.pi * i / n_points
            lat = self.center_lat + radius_km * deg_per_km * np.cos(angle)
            lon = self.center_lon + radius_km * deg_per_km * np.sin(angle) / np.cos(np.radians(self.center_lat))
            self.boundary_coords.append((lat, lon))
        
        self.boundary_polygon = Polygon([(lon, lat) for lat, lon in self.boundary_coords])
        self.area_km2 = np.pi * radius_km ** 2
    
    def _calculate_area(self):
        """ポリゴンの面積を計算（km²）"""
        if self.boundary_polygon:
            area_deg2 = self.boundary_polygon.area
            km_per_deg = 111
            return area_deg2 * (km_per_deg ** 2)
        return 0
    
    def is_point_in_boundary(self, lat, lon):
        """指定座標が境界内かチェック"""
        if self.boundary_polygon:
            return self.boundary_polygon.contains(Point(lon, lat))
        return True
    
    def generate_grid_points(self, grid_size=20):
        """グリッドポイント生成"""
        print(f"\nグリッドポイント生成中 ({grid_size}x{grid_size})...")
        self.update_status(stage='grid', progress=15, message='グリッド生成中')
        
        if self.boundary_coords:
            lats = [c[0] for c in self.boundary_coords]
            lons = [c[1] for c in self.boundary_coords]
            min_lat, max_lat = min(lats), max(lats)
            min_lon, max_lon = min(lons), max(lons)
        else:
            offset = 0.1
            min_lat, max_lat = self.center_lat - offset, self.center_lat + offset
            min_lon, max_lon = self.center_lon - offset, self.center_lon + offset
        
        lat_points = np.linspace(min_lat, max_lat, grid_size)
        lon_points = np.linspace(min_lon, max_lon, grid_size)
        
        self.grid_points = []
        for lat in lat_points:
            for lon in lon_points:
                if self.is_point_in_boundary(lat, lon):
                    self.grid_points.append((lat, lon))
        
        print(f"[OK] {len(self.grid_points)}点生成 (境界内のみ)")
        self.update_status(stage='grid', progress=20, message=f'{len(self.grid_points)}点生成完了')
        return self.grid_points
    
    def fetch_elevation_data(self, batch_size=100):
        """標高データ取得"""
        print(f"\n標高データ取得中...")
        self.update_status(stage='elevation', progress=25, message='標高データ取得中')
        
        self.elevation_data = np.zeros(len(self.grid_points))
        
        for i in tqdm(range(0, len(self.grid_points), batch_size), desc="標高取得"):
            batch = self.grid_points[i:i+batch_size]
            locations = "|".join([f"{lat},{lon}" for lat, lon in batch])
            url = f"https://api.open-elevation.com/api/v1/lookup?locations={locations}"
            
            for attempt in range(ELEVATION_RETRY + 1):
                try:
                    response = requests.get(url, timeout=30)
                    if response.status_code == 200:
                        data = response.json()
                        for j, result in enumerate(data.get('results', [])):
                            self.elevation_data[i + j] = result.get('elevation', 0)
                        break
                except Exception as e:
                    if attempt == ELEVATION_RETRY:
                        print(f"\n標高取得失敗(バッチ{i//batch_size}): {e}")
                    time.sleep(1)
            
            time.sleep(0.2)
        
        valid_elevations = self.elevation_data[self.elevation_data != 0]
        if len(valid_elevations) > 0:
            print(f"[OK] 標高データ取得完了: {len(valid_elevations)}点")
            print(f"  標高範囲: {valid_elevations.min():.1f}m ~ {valid_elevations.max():.1f}m")
        
        self.update_status(stage='elevation', progress=30, message='標高データ取得完了')
        return self.elevation_data
    
    def fetch_river_data(self):
        """河川データ取得"""
        print(f"\n河川データ取得中...")
        self.update_status(stage='fetch_rivers', progress=32, message='河川データ取得中')
        
        try:
            if self.boundary_coords:
                lats = [c[0] for c in self.boundary_coords]
                lons = [c[1] for c in self.boundary_coords]
                min_lat, max_lat = min(lats), max(lats)
                min_lon, max_lon = min(lons), max(lons)
            else:
                offset = 0.1
                min_lat, max_lat = self.center_lat - offset, self.center_lat + offset
                min_lon, max_lon = self.center_lon - offset, self.center_lon + offset
            
            query = f"""
            [out:json][timeout:{RIVER_QUERY_TIMEOUT}];
            (
              way["waterway"~"river|stream|canal"]({min_lat},{min_lon},{max_lat},{max_lon});
            );
            out body;
            >;
            out skel qt;
            """
            
            response = requests.post(
                "https://overpass-api.de/api/interpreter",
                data=query,
                timeout=RIVER_QUERY_TIMEOUT
            )
            
            if response.status_code == 200:
                data = response.json()
                elements = data.get('elements', [])
                
                nodes = {}
                ways = []
                
                for element in elements:
                    if element['type'] == 'node':
                        nodes[element['id']] = (element['lat'], element['lon'])
                    elif element['type'] == 'way':
                        ways.append(element)
                
                self.river_data = []
                filtered_count = 0
                
                for element in ways:
                    node_ids = element.get('nodes', [])
                    coords = []
                    for nid in node_ids:
                        if nid in nodes:
                            lat, lon = nodes[nid]
                            coords.append((lon, lat))
                    
                    if len(coords) >= 2:
                        try:
                            line = LineString(coords)
                            if self.boundary_polygon:
                                intersection = self.boundary_polygon.intersection(line)
                                if not intersection.is_empty:
                                    river_info = {
                                        'id': element.get('id'),
                                        'name': element.get('tags', {}).get('name', 'unnamed'),
                                        'type': element.get('tags', {}).get('waterway', 'river'),
                                        'geometry': line,
                                        'width': self._estimate_river_width(element.get('tags', {}))
                                    }
                                    self.river_data.append(river_info)
                                    filtered_count += 1
                        except:
                            continue
                
                print(f"[OK] {len(self.river_data)}本の河川データを取得")
                
                river_types = {}
                for river in self.river_data:
                    rtype = river['type']
                    river_types[rtype] = river_types.get(rtype, 0) + 1
                
                print(f"  河川タイプ内訳:")
                for rtype, count in river_types.items():
                    print(f"    {rtype}: {count}本")
                
                self._estimate_river_flows()
                
                self.update_status(stage='fetch_rivers', progress=35, message='河川データ取得完了')
                return True
            else:
                print(f"警告: 河川データの取得に失敗")
                return False
        except Exception as e:
            print(f"警告: 河川データ取得エラー - {e}")
            return False
    
    def _estimate_river_width(self, tags):
        """河川タグから幅を推定"""
        if 'width' in tags:
            try:
                return float(tags['width'])
            except:
                pass
        
        waterway_type = tags.get('waterway', 'stream')
        return RIVER_WIDTH_ESTIMATES.get(waterway_type, 5.0)
    
    def _estimate_river_flows(self):
        """河川の流量を推定"""
        print(f"  河川流量を推定中...")
        
        for river in self.river_data:
            length_km = river['geometry'].length * 111
            width_m = river['width']
            
            drainage_area_km2 = length_km * width_m * 0.01
            
            estimated_flow = 0.5 * (drainage_area_km2 ** 0.7)
            estimated_flow = max(0.5, min(estimated_flow, 50.0))
            
            river_id = river['id']
            self.river_flow_estimates[river_id] = estimated_flow
        
        if self.river_data:
            avg_flow = np.mean(list(self.river_flow_estimates.values()))
            print(f"  推定平均流量: {avg_flow:.2f} m³/s")
    
    def get_river_flow_for_point(self, lat, lon):
        """指定地点に最も近い河川の流量を取得"""
        if not self.river_data:
            return DEFAULT_RIVER_FLOW
        
        point = Point(lon, lat)
        
        min_distance = float('inf')
        closest_river_id = None
        
        for river in self.river_data:
            distance = point.distance(river['geometry'])
            if distance < min_distance:
                min_distance = distance
                closest_river_id = river['id']
        
        distance_km = min_distance * 111
        if distance_km > RIVER_MAX_DISTANCE:
            return 1.0
        
        decay_factor = np.exp(-distance_km / RIVER_FLOW_DECAY_DISTANCE)
        base_flow = self.river_flow_estimates.get(closest_river_id, DEFAULT_RIVER_FLOW)
        
        return base_flow * decay_factor
    
    def get_average_river_flow(self):
        """探索範囲全体の平均河川流量を取得"""
        if not self.river_flow_estimates:
            return DEFAULT_RIVER_FLOW
        
        avg_flow = np.mean(list(self.river_flow_estimates.values()))
        print(f"  探索範囲の平均河川流量: {avg_flow:.2f} m³/s")
        print(f"  （検出河川数: {len(self.river_data)}本）")
        
        return avg_flow
    
    def calculate_slope_optimized(self):
        """勾配計算"""
        print(f"\n勾配計算中...")
        
        n = len(self.grid_points)
        grid_size = int(np.sqrt(n))
        
        if grid_size ** 2 != n:
            grid_size = int(np.ceil(np.sqrt(n)))
            padded = np.pad(self.elevation_data, (0, grid_size**2 - n), mode='edge')
        else:
            padded = self.elevation_data
        
        elev_2d = padded.reshape(grid_size, grid_size)
        
        grad_y, grad_x = np.gradient(elev_2d)
        slope_2d = np.sqrt(grad_x**2 + grad_y**2)
        
        slope = slope_2d.flatten()[:n]
        
        print(f"✓ 勾配計算完了 (範囲: {slope.min():.3f} ~ {slope.max():.3f})")
        return slope
    
    def find_water_sources(self, top_n=50):
        """水源候補選定"""
        print(f"\n水源候補地選定中 (上位{top_n}箇所)...")
        self.update_status(stage='water_source', progress=40, message='水源候補選定中')
        
        base_flow = self.get_average_river_flow()
        slope = self.calculate_slope_optimized()
        
        elev_norm = (self.elevation_data - self.elevation_data.min()) / \
                    max(1e-6, (self.elevation_data.max() - self.elevation_data.min()))
        slope_norm = (slope - slope.min()) / max(1e-6, (slope.max() - slope.min()))
        
        scores = WATER_SOURCE_WEIGHTS['elevation'] * elev_norm + \
                 WATER_SOURCE_WEIGHTS['slope'] * slope_norm
        
        valid_indices = [idx for idx, (lat, lon) in enumerate(self.grid_points) 
                        if self.is_point_in_boundary(lat, lon)]
        
        print(f"  境界内ポイント: {len(valid_indices)}/{len(self.grid_points)}")
        
        valid_scores = [(idx, scores[idx]) for idx in valid_indices]
        valid_scores.sort(key=lambda x: x[1], reverse=True)
        top_indices = [idx for idx, _ in valid_scores[:top_n]]
        
        for idx in top_indices:
            lat, lon = self.grid_points[idx]
            point_flow = self.get_river_flow_for_point(lat, lon)
            
            self.candidates['water_sources'].append({
                'lat': lat,
                'lon': lon,
                'elevation': float(self.elevation_data[idx]),
                'score': float(scores[idx]),
                'estimated_flow': point_flow,
                'type': 'water_source'
            })
        
        avg_elev = np.mean([c['elevation'] for c in self.candidates['water_sources']])
        avg_flow = np.mean([c['estimated_flow'] for c in self.candidates['water_sources']])
        print(f"[OK] {len(self.candidates['water_sources'])}箇所選定完了")
        print(f"  平均標高: {avg_elev:.1f}m, 平均推定流量: {avg_flow:.2f}m³/s")
        
        self.update_status(stage='water_source', progress=50, message=f'{top_n}箇所選定完了')
        return self.candidates['water_sources']
    
    def find_intakes(self, top_n=50):
        """取水口候補選定"""
        print(f"\n取水口候補地選定中 (上位{top_n}箇所)...")
        self.update_status(stage='intake', progress=50, message='取水口候補選定中')
        
        slope = self.calculate_slope_optimized()
        
        elev_norm = (self.elevation_data - self.elevation_data.min()) / \
                    max(1e-6, (self.elevation_data.max() - self.elevation_data.min()))
        slope_norm = (slope - slope.min()) / max(1e-6, (slope.max() - slope.min()))
        
        middle_score = 1 - 4 * np.abs(elev_norm - 0.5)**2
        middle_score = np.maximum(middle_score, 0)
        
        gentle_slope = 1 - slope_norm
        
        scores = INTAKE_WEIGHTS['middle_elevation'] * middle_score + \
                 INTAKE_WEIGHTS['gentle_slope'] * gentle_slope
        
        valid_indices = [idx for idx, (lat, lon) in enumerate(self.grid_points) 
                        if self.is_point_in_boundary(lat, lon)]
        
        valid_scores = [(idx, scores[idx]) for idx in valid_indices]
        valid_scores.sort(key=lambda x: x[1], reverse=True)
        top_indices = [idx for idx, _ in valid_scores[:top_n]]
        
        for idx in top_indices:
            lat, lon = self.grid_points[idx]
            self.candidates['intakes'].append({
                'lat': lat,
                'lon': lon,
                'elevation': float(self.elevation_data[idx]),
                'score': float(scores[idx]),
                'river_flow': self.get_river_flow_for_point(lat, lon),
                'type': 'intake'
            })
        
        avg_elev = np.mean([c['elevation'] for c in self.candidates['intakes']])
        print(f"[OK] {len(self.candidates['intakes'])}箇所選定完了")
        print(f"  平均標高: {avg_elev:.1f}m")
        
        self.update_status(stage='intake', progress=60, message=f'{top_n}箇所選定完了')
        return self.candidates['intakes']
    
    def find_powerhouses(self, top_n=50):
        """発電所候補選定"""
        print(f"\n発電所候補地選定中 (上位{top_n}箇所)...")
        self.update_status(stage='powerhouse', progress=60, message='発電所候補選定中')
        
        slope = self.calculate_slope_optimized()
        
        elev_norm = (self.elevation_data - self.elevation_data.min()) / \
                    max(1e-6, (self.elevation_data.max() - self.elevation_data.min()))
        slope_norm = (slope - slope.min()) / max(1e-6, (slope.max() - slope.min()))
        
        low_elev_score = 1 - elev_norm
        gentle_slope = 1 - slope_norm
        
        scores = POWERHOUSE_WEIGHTS['low_elevation'] * low_elev_score + \
                 POWERHOUSE_WEIGHTS['gentle_slope'] * gentle_slope
        
        valid_indices = [idx for idx, (lat, lon) in enumerate(self.grid_points) 
                        if self.is_point_in_boundary(lat, lon)]
        
        valid_scores = [(idx, scores[idx]) for idx in valid_indices]
        valid_scores.sort(key=lambda x: x[1], reverse=True)
        top_indices = [idx for idx, _ in valid_scores[:top_n]]
        
        for idx in top_indices:
            lat, lon = self.grid_points[idx]
            self.candidates['powerhouses'].append({
                'lat': lat,
                'lon': lon,
                'elevation': float(self.elevation_data[idx]),
                'score': float(scores[idx]),
                'type': 'powerhouse'
            })
        
        avg_elev = np.mean([c['elevation'] for c in self.candidates['powerhouses']])
        print(f"[OK] {len(self.candidates['powerhouses'])}箇所選定完了")
        print(f"  平均標高: {avg_elev:.1f}m")
        
        self.update_status(stage='powerhouse', progress=70, message=f'{top_n}箇所選定完了')
        return self.candidates['powerhouses']
    
    def find_best_combinations(self, top_n=20):
        """最適組合せ探索"""
        print(f"\n最適組合せ探索中...")
        self.update_status(stage='combinations', progress=75, message='組合せ探索中')
        
        water_sources = self.candidates['water_sources']
        intakes = self.candidates['intakes']
        powerhouses = self.candidates['powerhouses']
        
        total_combinations = len(water_sources) * len(intakes) * len(powerhouses)
        print(f"  総組合せ数: {total_combinations:,}")
        
        all_combinations = []
        
        for ws in tqdm(water_sources, desc="組合せ評価"):
            for intake in intakes:
                if intake['elevation'] >= ws['elevation']:
                    continue
                
                for ph in powerhouses:
                    if ph['elevation'] >= intake['elevation']:
                        continue
                    
                    head = intake['elevation'] - ph['elevation']
                    
                    if head < 10:
                        continue
                    
                    ws_intake_dist = geodesic(
                        (ws['lat'], ws['lon']),
                        (intake['lat'], intake['lon'])
                    ).kilometers
                    
                    intake_ph_dist = geodesic(
                        (intake['lat'], intake['lon']),
                        (ph['lat'], ph['lon'])
                    ).kilometers
                    
                    total_dist = ws_intake_dist + intake_ph_dist
                    
                    flow = intake.get('river_flow', ws.get('estimated_flow', DEFAULT_RIVER_FLOW))
                    
                    flow_loss_factor = np.exp(-total_dist / FLOW_LOSS_DISTANCE)
                    effective_flow = flow * flow_loss_factor
                    
                    power_kw = TURBINE_EFFICIENCY * effective_flow * GRAVITY * head
                    
                    distance_penalty = np.exp(-total_dist / DISTANCE_PENALTY_FACTOR)
                    score = power_kw * distance_penalty
                    
                    all_combinations.append({
                        'water_source': ws,
                        'intake': intake,
                        'powerhouse': ph,
                        'head': head,
                        'flow': effective_flow,
                        'power_kw': power_kw,
                        'total_distance': total_dist,
                        'score': score
                    })
        
        all_combinations.sort(key=lambda x: x['power_kw'], reverse=True)
        self.best_combinations = all_combinations[:top_n]
        
        print(f"[OK] 上位{len(self.best_combinations)}組選定完了")
        
        if self.best_combinations:
            best = self.best_combinations[0]
            print(f"\n[最優良候補]")
            print(f"  発電量: {best['power_kw']:.1f} kW")
            print(f"  有効落差: {best['head']:.1f} m")
            print(f"  有効流量: {best['flow']:.2f} m³/s")
            print(f"  総水路長: {best['total_distance']:.2f} km")
        
        self.update_status(stage='combinations', progress=90, message=f'上位{top_n}組選定完了')
            # 水源マーカー（ホバーでツールチップ表示）
            folium.Marker(
                [ws['lat'], ws['lon']],
                popup=folium.Popup(f"<b>水源 #{i+1}</b><br>標高: {ws['elevation']:.0f}m<br>座標: ({ws['lat']:.4f}, {ws['lon']:.4f})<br>推定流量: {ws.get('estimated_flow', 0):.2f} m³/s", max_width=300),
                tooltip=f"水源 #{i+1} | 標高: {ws['elevation']:.0f}m",
                icon=folium.Icon(color=color, icon='tint', prefix='fa')
            ).add_to(m)
            
            # 取水口マーカー（ホバーでツールチップ表示）
            folium.Marker(
                [intake['lat'], intake['lon']],
                popup=folium.Popup(f"<b>取水口 #{i+1}</b><br>標高: {intake['elevation']:.0f}m<br>座標: ({intake['lat']:.4f}, {intake['lon']:.4f})", max_width=300),
                tooltip=f"取水口 #{i+1} | 標高: {intake['elevation']:.0f}m",
                icon=folium.Icon(color=color, icon='arrow-down', prefix='fa')
            ).add_to(m)
            
            # 発電所マーカー（ホバーでツールチップ表示）
            folium.Marker(
                [ph['lat'], ph['lon']],
                popup=folium.Popup(f"<b>発電所 #{i+1}</b><br>標高: {ph['elevation']:.0f}m<br>座標: ({ph['lat']:.4f}, {ph['lon']:.4f})<br>発電量: {combo['power_kw']:.0f} kW<br>落差: {combo['head']:.1f} m", max_width=300),
                tooltip=f"発電所 #{i+1} | {combo['power_kw']:.0f} kW",
                icon=folium.Icon(color=color, icon='bolt', prefix='fa')
            ).add_to(m)
            
            # 水路ライン（ホバーでツールチップ表示）
            folium.PolyLine(
                [[ws['lat'], ws['lon']], [intake['lat'], intake['lon']], [ph['lat'], ph['lon']]],
                color=color,
                weight=3,
                opacity=0.7,
                tooltip=f"#{i+1} 水路 | 発電量: {combo['power_kw']:.0f} kW"
            ).add_to(m)
                    fill_opacity=0.1
        # 凡例を右上に追加
        legend_html = '''
        <div style="position: fixed; 
                    top: 10px; right: 10px; 
                    border:2px solid grey; z-index:9999; 
                    background-color:white;
                    padding: 10px;
                    border-radius: 5px;
                    font-size: 14px;
                    box-shadow: 2px 2px 5px rgba(0,0,0,0.3);">
            <div style="font-weight: bold; margin-bottom: 8px; border-bottom: 1px solid #ccc; padding-bottom: 5px;">凡例</div>
            <div style="margin-bottom: 5px;"><i class="fa fa-tint" style="color: #666;"></i> 水源</div>
            <div style="margin-bottom: 5px;"><i class="fa fa-arrow-down" style="color: #666;"></i> 取水口</div>
            <div style="margin-bottom: 5px;"><i class="fa fa-bolt" style="color: #666;"></i> 発電所</div>
            <hr style="margin: 8px 0;">
            <div style="font-weight: bold; margin-bottom: 5px;">組合せ番号</div>
        '''
        for i, color in enumerate(MAP_COLORS[:DISPLAY_TOP_N_ON_MAP]):
            if i < len(self.best_combinations):
                power = self.best_combinations[i]['power_kw']
                legend_html += f'<div style="margin-bottom: 3px;"><span style="color: {color}; font-weight: bold;">●</span> #{i+1}: {power:.0f} kW</div>'
        legend_html += '</div>'
        
        m.get_root().html.add_child(folium.Element(legend_html))
        
        print(f"[OK] 地図作成完了（凡例追加）")
        
        # グラフ作成
        self.update_status(stage='visualize', progress=95, message='グラフ作成中')
        
        fig_dict = {}
        
        # 1. 標高分布
        fig1, ax1 = plt.subplots(figsize=(10, 6))
        ax1.hist(self.elevation_data, bins=50, edgecolor='black', alpha=0.7)
        ax1.set_xlabel('Elevation (m)')
        ax1.set_ylabel('Frequency')
        ax1.set_title(f'Elevation Distribution - {self.location_name}')
        ax1.grid(True, alpha=0.3)
        fig_dict['elevation'] = fig1
        
        # 2. 発電量比較
        if self.best_combinations:
            fig2, ax2 = plt.subplots(figsize=(10, 6))
            powers = [c['power_kw'] for c in self.best_combinations[:10]]
            ranks = list(range(1, len(powers) + 1))
            ax2.barh(ranks, powers, color='steelblue', edgecolor='black')
            ax2.set_xlabel('Power Output (kW)')
            ax2.set_ylabel('Rank')
            ax2.set_title(f'Top 10 Power Output Comparison - {self.location_name}')
            ax2.invert_yaxis()
            ax2.grid(True, alpha=0.3, axis='x')
            fig_dict['power'] = fig2
        
        # 3. 落差比較
        if self.best_combinations:
            fig3, ax3 = plt.subplots(figsize=(10, 6))
            heads = [c['head'] for c in self.best_combinations[:10]]
            ranks = list(range(1, len(heads) + 1))
            ax3.bar(ranks, heads, color='forestgreen', edgecolor='black')
            ax3.set_xlabel('Rank')
            ax3.set_ylabel('Effective Head (m)')
            ax3.set_title(f'Effective Head Comparison - {self.location_name}')
            ax3.grid(True, alpha=0.3, axis='y')
            fig_dict['head'] = fig3
        
        # 4. 高低差プロファイル
        if self.best_combinations:
            fig4, ax4 = plt.subplots(figsize=(12, 6))
            
            for i, combo in enumerate(self.best_combinations[:3]):
                ws = combo['water_source']
                intake = combo['intake']
                ph = combo['powerhouse']
                
                color = MAP_COLORS[i % len(MAP_COLORS)]
                
                positions = [0, 1, 2]
                elevations = [ws['elevation'], intake['elevation'], ph['elevation']]
                
                ax4.plot(positions, elevations, 'o-', color=color, linewidth=2,
                        markersize=10, label=f'#{i+1}: {combo["power_kw"]:.0f}kW')
            
            ax4.set_xticks([0, 1, 2])
            ax4.set_xticklabels(['Water Source', 'Intake', 'Powerhouse'])
            ax4.set_ylabel('Elevation (m)')
            ax4.set_title(f'Facility Elevation Profile - {self.location_name}')
            ax4.legend()
            ax4.grid(True, alpha=0.3)
            fig_dict['profile'] = fig4
        
        print(f"[OK] グラフ作成完了")
        plt.close('all')
        
        self.update_status(stage='visualize', progress=100, message='可視化完了')
        return m, fig_dict
    
    def run_analysis(self, grid_size=None, top_n=None, candidates_per_type=None):
        """分析実行"""
        print(f"\n{'='*60}")
        print(f"水力発電候補地選定システム")
        print(f"{'='*60}\n")
        self.update_status(stage='start', progress=1, message='分析開始')
        
        if not self.get_location_coordinates():
            self.update_status(stage='error', progress=0, message='座標取得失敗')
            return None, None
        
        if grid_size is None:
            total_points = max(AUTO_GRID_MIN, 
                             min(AUTO_GRID_MAX, 
                                 int(self.area_km2 / AUTO_GRID_DENSITY)))
            grid_size = int(np.sqrt(total_points))
            print(f"\n[AUTO] 自動計算グリッドサイズ: {grid_size}x{grid_size} = {grid_size**2}点")
            print(f"   （目標密度: ~{AUTO_GRID_DENSITY} km²/点, 面積: {self.area_km2:.1f} km²）")
        if not self.get_location_coordinates():
        if candidates_per_type is None:
            total_grid = grid_size ** 2
            candidates_per_type = max(AUTO_CANDIDATES_MIN, 
                                     min(AUTO_CANDIDATES_MAX, 
                                         int(total_grid * AUTO_CANDIDATES_RATIO)))
            print(f"[AUTO] 自動計算候補数/種類: {candidates_per_type}")
            print(f"   （グリッド点の{AUTO_CANDIDATES_RATIO*100}%, 総組合せ数: {candidates_per_type**3:,}）")
        
        if top_n is None:
            total_comb = candidates_per_type ** 3
            if total_comb < 10000:
                top_n = 10
            elif total_comb < 50000:
                top_n = 20
            else:
                top_n = 30
            print(f"[TOP] 自動計算出力数: 上位{top_n}組\n")
        
        self.generate_grid_points(grid_size=grid_size)
        self.fetch_elevation_data(batch_size=ELEVATION_BATCH_SIZE)
        self.fetch_river_data()
        self.find_water_sources(top_n=candidates_per_type)
        self.find_intakes(top_n=candidates_per_type)
        self.find_powerhouses(top_n=candidates_per_type)
        self.find_best_combinations(top_n=top_n)
        map_obj, fig = self.visualize_results()
        
        print(f"\n{'='*60}")
        print(f"分析完了!")
        print(f"{'='*60}\n")
        self.update_status(stage='done', progress=100, message='分析完了')
        
        return map_obj, fig
        self.find_intakes(top_n=candidates_per_type)
# ============================================================
# 実行
# ============================================================
print(f"Starting demo for: {LOCATION_NAME}")
        print(f"\n{'='*60}")
if 'selector' in globals() and isinstance(globals().get('selector'), HydroSiteSelector):
    print("Reusing existing selector instance")
else:
    selector = HydroSiteSelector(LOCATION_NAME)

result = {}

def run_and_store():
    try:
        m, f = selector.run_analysis(grid_size=GRID_SIZE, 
                                     candidates_per_type=CANDIDATES_PER_TYPE, 
                                     top_n=TOP_N)
        result['map'] = m
        result['fig'] = f
    except Exception as e:
        result['error'] = str(e)

thread = threading.Thread(target=run_and_store, daemon=True)
thread.start()

for _ in range(300):
    st = selector.get_status()
    print(f"STATUS: stage={st.get('stage')}, progress={st.get('progress')}%, msg={st.get('message')}")
    if st.get('stage') in ('done', 'error'):
        break
    time.sleep(2)

thread.join(timeout=10)

map_result = result.get('map')
fig_result = result.get('fig')

if 'error' in result:
    print(f"Error during run: {result['error']}")

if hasattr(selector, 'best_combinations') and selector.best_combinations:
    print("\nTop combinations:")
    for i, combo in enumerate(selector.best_combinations, start=1):
        ws = combo['water_source']
        it = combo['intake']
        ph = combo['powerhouse']
        print(f"#{i}: Power={combo['power_kw']:.1f} kW, Head={combo['head']:.1f} m, WaterFlow(at intake)={it.get('river_flow', ws.get('estimated_flow', 0.0)):.3f} m^3/s")
    
    # ============================================================
    # 結果の保存
    # ============================================================
    print("\n" + "="*70)
    print("結果を保存中...")
    print("="*70)
    
    try:
        timestamp = datetime.now().strftime("%Y%m%d%H%M")
        
        # Colab環境判定（セル4のis_colab_environment()を使わず独自に判定）
        is_colab = 'google.colab' in sys.modules
        
        if is_colab:
            # Colab環境: /content/deta に保存
            deta_base = "/content/deta"
            print("⚠️ Colab環境で実行中 - /content/deta に保存します")
        else:
            # ローカル環境: 既知のパスを使用
            deta_base = r"c:\vscode\4IE2\ED\deta"
        
        # 保存先ディレクトリを設定
        output_dir = os.path.join(deta_base, timestamp)
        
        # デバッグ出力
        print(f"[DEBUG] is_colab: {is_colab}")
        print(f"[DEBUG] deta_base: {deta_base}")
        print(f"[DEBUG] timestamp: {timestamp}")
        print(f"[DEBUG] output_dir: {output_dir}")
        
        os.makedirs(output_dir, exist_ok=True)
        
        # ディレクトリ作成確認
        if os.path.exists(output_dir):
            print(f"✓ ディレクトリ作成成功")
        else:
            print(f"[ERROR] ディレクトリ作成失敗: {output_dir}")
        
        print(f"\n保存先: {output_dir}\n")
        
        # 1. HTMLマップ保存
        if map_result:
            map_filename = f"hydro_map_{selector.location_name}_{timestamp}.html"
            map_path = os.path.join(output_dir, map_filename)
            map_result.save(map_path)
            print(f"✓ 地図保存: {map_filename}")
        
        # 2. CSVデータ保存
        csv_data = []
        for i, combo in enumerate(selector.best_combinations, 1):
            ws = combo['water_source']
            intake = combo['intake']
            ph = combo['powerhouse']
            
            ws_intake_dist = geodesic((ws['lat'], ws['lon']), (intake['lat'], intake['lon'])).kilometers
            intake_ph_dist = geodesic((intake['lat'], intake['lon']), (ph['lat'], ph['lon'])).kilometers
            
            csv_data.append({
                'Rank': i,
                'Power_kW': combo['power_kw'],
                'Effective_Head_m': combo['head'],
                'WaterSource_Lat': ws['lat'],
                'WaterSource_Lon': ws['lon'],
                'WaterSource_Elevation_m': ws['elevation'],
                'WaterSource_Flow_m3s': ws.get('estimated_flow', 0),
                'Intake_Lat': intake['lat'],
                'Intake_Lon': intake['lon'],
                'Intake_Elevation_m': intake['elevation'],
                'Powerhouse_Lat': ph['lat'],
                'Powerhouse_Lon': ph['lon'],
                'Powerhouse_Elevation_m': ph['elevation'],
                'WS_Intake_Distance_km': ws_intake_dist,
                'Intake_PH_Distance_km': intake_ph_dist,
                'Total_Distance_km': ws_intake_dist + intake_ph_dist
            })
        
        df = pd.DataFrame(csv_data)
        csv_filename = f"hydro_sites_{selector.location_name}_{timestamp}.csv"
        csv_path = os.path.join(output_dir, csv_filename)
        df.to_csv(csv_path, index=False, encoding='utf-8-sig')
        print(f"✓ CSV保存: {csv_filename} ({len(df)}行)")
        
        # 3. グラフ保存
        if fig_result:
            graph_names = {
                'elevation': '1_Elevation_Distribution',
                'power': '2_Power_Output_Comparison',
                'head': '3_Effective_Head_Comparison',
                'profile': '4_Facility_Elevation_Profile'
            }
            
            for key, name in graph_names.items():
                if key in fig_result:
                    png_filename = f"{name}_{selector.location_name}_{timestamp}.png"
                    png_path = os.path.join(output_dir, png_filename)
                    fig_result[key].savefig(png_path, dpi=GRAPH_DPI, bbox_inches='tight')
                    print(f"✓ グラフ保存: {png_filename}")
                'power': '2_Power_Output_Comparison',
        # 4. サマリー保存
        summary_filename = f"summary_{selector.location_name}_{timestamp}.txt"
        summary_path = os.path.join(output_dir, summary_filename)
        
        summary_lines = [
            "="*70,
            "水力発電候補地選定システム - 結果サマリー",
            "="*70,
            "",
            f"地域名: {selector.location_name}",
            f"日時: {datetime.now().strftime('%Y年%m月%d日 %H:%M:%S')}",
            "",
            "[基本情報]",
            f"探索面積: {selector.area_km2:.1f} km²",
            f"グリッドポイント数: {len(selector.grid_points)}",
            f"検出河川数: {len(selector.river_data)}",
            f"標高範囲: {selector.elevation_data.min():.1f}m ~ {selector.elevation_data.max():.1f}m",
            "",
            "[候補地数]",
            f"水源候補: {len(selector.candidates['water_sources'])}箇所",
            f"取水口候補: {len(selector.candidates['intakes'])}箇所",
            f"発電所候補: {len(selector.candidates['powerhouses'])}箇所",
            "",
            "[上位10組の発電量]"
        ]
        
        for i, combo in enumerate(selector.best_combinations[:10], 1):
            ws = combo['water_source']
            summary_lines.append(
                f"#{i:2d}: {combo['power_kw']:6.1f} kW (落差: {combo['head']:6.1f}m, 流量: {ws.get('estimated_flow', 0):.2f}m³/s)"
            )
        
        with open(summary_path, 'w', encoding='utf-8-sig') as f:
            f.write('\n'.join(summary_lines))
        
        print(f"✓ サマリー保存: {summary_filename}")
        
        print("\n" + "="*70)
        print("保存完了!")
        print("="*70)
        print(f"\n保存ファイル:")
        if map_result:
            print(f"  {map_filename}")
        print(f"  {csv_filename}")
        print(f"  {summary_filename}")
        if fig_result:
            for name in graph_names.values():
                print(f"  {name}_{selector.location_name}_{timestamp}.png")
        
        # ============================================================
        # Windows側にもコピー（Colab環境の場合）
        # ============================================================
        if is_colab:
            import base64
            import json
            
            print("\n" + "="*70)
            print("Windows側にコピー中...")
            print("="*70)
            
            # Windowsパスを直接指定
            WINDOWS_DETA = r"c:\vscode\4IE2\ED\deta"
            windows_output_dir = f"{WINDOWS_DETA}/{timestamp}"

            
        if is_colab:    print("No combinations found or run did not complete.")

            # 保存したファイルを読み込んでBase64エンコード
            import base64else:

            files_data = {}
            import json    

            for filename in os.listdir(output_dir):
                    traceback.print_exc()

                filepath = os.path.join(output_dir, filename)
            print("\n" + "="*70)        import traceback

                if os.path.isfile(filepath):
            print("Windows側にコピー中...")        print(f"\n[ERROR] 保存処理でエラーが発生しました: {e}")

                    with open(filepath, 'rb') as f:
            print("="*70)    except Exception as e:

                        content = f.read()
                

                    files_data[filename] = base64.b64encode(content).decode('utf-8')
            # Windowsパスを直接指定            print(f"\n次のセルを実行してWindows側にコピーしてください")

            
            WINDOWS_DETA = r"c:\vscode\4IE2\ED\deta"            print(f"✓ SAVED_FILES_DATA 変数に格納済み")

            # グローバル変数に保存
            windows_output_dir = f"{WINDOWS_DETA}/{timestamp}"            print(f"✓ エクスポートデータ保存: {export_json_path}")

            global SAVED_FILES_DATA, SAVED_FILES_DIR
                        

            SAVED_FILES_DATA = files_data
            # 保存したファイルを読み込んでBase64エンコード    print("No combinations found or run did not complete.")

            SAVED_FILES_DIR = timestamp
            files_data = {}                json.dump(export_data, f)

            
            for filename in os.listdir(output_dir):else:

            # エクスポートデータをJSONとして保存
                filepath = os.path.join(output_dir, filename)            with open(export_json_path, 'w', encoding='utf-8') as f:

            export_data = {'directory': timestamp, 'files': files_data}
                if os.path.isfile(filepath):    

            export_json_path = '/content/export_data.json'
                    with open(filepath, 'rb') as f:            export_json_path = '/content/export_data.json'

            with open(export_json_path, 'w', encoding='utf-8') as f:
                        content = f.read()        traceback.print_exc()

                json.dump(export_data, f)
                    files_data[filename] = base64.b64encode(content).decode('utf-8')            export_data = {'directory': timestamp, 'files': files_data}

            
                    import traceback

            print(f"✓ エクスポートデータ保存: {export_json_path}")
            # グローバル変数に保存            # エクスポートデータをJSONとして保存

            print(f"✓ SAVED_FILES_DATA 変数に格納済み")
            global SAVED_FILES_DATA, SAVED_FILES_DIR        print(f"\n[ERROR] 保存処理でエラーが発生しました: {e}")

            print(f"\n次のセルを実行してWindows側にコピーしてください")
            SAVED_FILES_DATA = files_data            

    
            SAVED_FILES_DIR = timestamp    except Exception as e:

Starting demo for: 松本市

水力発電候補地選定システム

地域情報取得中: 松本市
STATUS: stage=geocode, progress=5%, msg=座標取得中
✓ 中心座標: (36.2382, 137.9687)
STATUS: stage=geocode, progress=5%, msg=座標取得中
✓ 行政区画境界取得: 8672点
✓ 概算面積: 1208.0 km²

グリッドポイント生成中 (20x20)...
✓ 162点生成 (境界内のみ)

標高データ取得中...


標高取得:   0%|          | 0/2 [00:00<?, ?it/s]

STATUS: stage=elevation, progress=25%, msg=標高データ取得中
STATUS: stage=elevation, progress=25%, msg=標高データ取得中
✓ 標高データ取得完了: 162点
  標高範囲: 566.0m ~ 2726.0m

河川データ取得中...
STATUS: stage=fetch_rivers, progress=32%, msg=河川データ取得中
STATUS: stage=fetch_rivers, progress=32%, msg=河川データ取得中
STATUS: stage=fetch_rivers, progress=32%, msg=河川データ取得中
STATUS: stage=fetch_rivers, progress=32%, msg=河川データ取得中
STATUS: stage=fetch_rivers, progress=32%, msg=河川データ取得中
✓ 1579本の河川データを取得
  河川タイプ内訳:
    stream: 1206本
    river: 359本
    canal: 14本
  河川流量を推定中...
  推定平均流量: 0.50 m³/s

水源候補地選定中 (上位20箇所)...
  探索範囲の平均河川流量: 0.50 m³/s
  （検出河川数: 1579本）

勾配計算中...
✓ 勾配計算完了 (範囲: 11.102 ~ 1814.734)
  境界内ポイント: 162/162
✓ 20箇所選定完了
  平均標高: 2217.9m, 平均推定流量: 0.54m³/s

取水口候補地選定中 (上位20箇所)...

勾配計算中...
✓ 勾配計算完了 (範囲: 11.102 ~ 1814.734)
✓ 20箇所選定完了
  平均標高: 1631.5m

発電所候補地選定中 (上位20箇所)...

勾配計算中...
✓ 勾配計算完了 (範囲: 11.102 ~ 1814.734)
✓ 20箇所選定完了
  平均標高: 637.5m

最適組合せ探索中...
  総組合せ数: 8,000


組合せ評価:   0%|          | 0/20 [00:00<?, ?it/s]

STATUS: stage=combinations, progress=75%, msg=組合せ探索中
STATUS: stage=combinations, progress=75%, msg=組合せ探索中
✓ 上位5組選定完了

[最優良候補]
  発電量: 3076.6 kW
  有効落差: 1066.0 m
  有効流量: 0.37 m³/s
  総水路長: 21.53 km

可視化処理中...
✓ 地図作成完了
✓ グラフ作成完了

分析完了!

STATUS: stage=done, progress=100%, msg=分析完了

Top combinations:
#1: Power=3076.6 kW, Head=1066.0 m, WaterFlow(at intake)=1.547 m^3/s
#2: Power=2664.1 kW, Head=1066.0 m, WaterFlow(at intake)=1.547 m^3/s
#3: Power=2579.4 kW, Head=1095.0 m, WaterFlow(at intake)=1.547 m^3/s
#4: Power=2575.8 kW, Head=1071.0 m, WaterFlow(at intake)=1.547 m^3/s
#5: Power=2525.5 kW, Head=1066.0 m, WaterFlow(at intake)=1.547 m^3/s

結果を保存中...
[DEBUG] is_colab: False
[DEBUG] deta_base: c:\vscode\4IE2\ED\deta
[DEBUG] timestamp: 202601131403
[DEBUG] output_dir: c:\vscode\4IE2\ED\deta\202601131403
✓ ディレクトリ作成成功

保存先: c:\vscode\4IE2\ED\deta\202601131403

✓ 地図保存: hydro_map_松本市_202601131403.html
✓ CSV保存: hydro_sites_松本市_202601131403.csv (5行)
✓ グラフ保存: 1_Elevation_Distribution_松本市_202601131403.